In [ ]:
import json
import os
import glob
import pandas as pd
import sys
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.append('..')
from utils.analysis_utils import extract_verdict
# Define constants
WHO_MADE_THIS_EXPERIMENT_DIR = '../monitor_responses/who_made_this/'


In [ ]:
# Load the data
all_loaded_data = []

if not os.path.exists(WHO_MADE_THIS_EXPERIMENT_DIR):
    print(f"Error: Experiment directory not found: {WHO_MADE_THIS_EXPERIMENT_DIR}")
else:
    task_names = [d for d in os.listdir(WHO_MADE_THIS_EXPERIMENT_DIR) if os.path.isdir(os.path.join(WHO_MADE_THIS_EXPERIMENT_DIR, d))]

    for task_name in task_names:
        task_path = os.path.join(WHO_MADE_THIS_EXPERIMENT_DIR, task_name)

        # 1. Load solution files
        solution_dir_path = os.path.join(task_path, 'solution')
        if os.path.isdir(solution_dir_path):
            monitor_model_dirs_solution = [d for d in os.listdir(solution_dir_path) if os.path.isdir(os.path.join(solution_dir_path, d))]
            for monitor_model in monitor_model_dirs_solution:
                # The generation script creates a specific file name for solutions
                solution_file_name = f"{task_name}_solution.json"
                file_path = os.path.join(solution_dir_path, monitor_model, solution_file_name)
                if os.path.exists(file_path):
                    try:
                        with open(file_path, 'r') as f:
                            data = json.load(f)
                            all_loaded_data.append({
                                'task_name': task_name,
                                'type': 'solution',
                                'monitor_model': monitor_model,
                                'agent_model': None, # No specific agent model for solutions in this context
                                'transcript_type_folder': 'solution',
                                'original_filename': os.path.basename(file_path),
                                'data': data
                            })
                    except json.JSONDecodeError:
                        print(f"Error decoding JSON from {file_path}")
                    except Exception as e:
                        print(f"Error loading {file_path}: {e}")
                # else: # Optionally, could glob here if filenames are not always fixed
                #    print(f"Expected solution file not found: {file_path}")

        # 2. Load transcript response files
        # Transcript types as used in the generation script
        transcript_type_folders = ['transcripts', 'transcripts_no_hint', 'transcripts_no_secondary_task']
        for tt_folder in transcript_type_folders:
            transcript_type_path = os.path.join(task_path, tt_folder)
            if os.path.isdir(transcript_type_path):
                # Discover agent model directories within this transcript type folder
                agent_model_names = [d for d in os.listdir(transcript_type_path) if os.path.isdir(os.path.join(transcript_type_path, d))]
                for agent_model_name in agent_model_names:
                    current_agent_model_path = os.path.join(transcript_type_path, agent_model_name)
                    # Discover monitor model directories within this agent model folder
                    monitor_model_names = [d for d in os.listdir(current_agent_model_path) if os.path.isdir(os.path.join(current_agent_model_path, d))]
                    for monitor_model_name in monitor_model_names:
                        current_monitor_path = os.path.join(current_agent_model_path, monitor_model_name)
                        json_files = glob.glob(os.path.join(current_monitor_path, '*.json'))
                        for file_path in json_files:
                            if os.path.exists(file_path):
                                try:
                                    with open(file_path, 'r') as f:
                                        data = json.load(f)
                                        all_loaded_data.append({
                                            'task_name': task_name,
                                            'type': 'transcript_response',
                                            'monitor_model': monitor_model_name,
                                            'agent_model': agent_model_name, # Agent that generated the original transcript
                                            'transcript_type_folder': tt_folder,
                                            'original_filename': os.path.basename(file_path),
                                            'data': data
                                        })
                                except json.JSONDecodeError:
                                    print(f"Error decoding JSON from {file_path}")
                                except Exception as e:
                                    print(f"Error loading {file_path}: {e}")

    print(f"Loaded {len(all_loaded_data)} data entries.")

# For quick verification, let's print the number of items and the first item if any
if all_loaded_data:
    print("First loaded item:")
    # Pretty print the first item for better readability if it's complex
    try:
        print(json.dumps(all_loaded_data[0], indent=2))
    except TypeError as e:
        print(f"Could not serialize first item for printing: {e}")
        print(all_loaded_data[0])
elif os.path.exists(WHO_MADE_THIS_EXPERIMENT_DIR):
    print("No data was loaded, but the experiment directory exists. Check paths and file structures.")
else:
    # This case is already handled by the check at the beginning of the cell
    pass 


In [ ]:
def extract_verdict_string(text, tag_type):
    """Extract text between <tag_type> and </tag_type> tags.
    
    Args:
        text (str): Text to search for tags
        tag_type (str): Type of tag to look for (e.g. 'verdict' or 'score')
        
    Returns:
        str: Text between tags, or None if not found
    """
    import re
    pattern = f"<{tag_type}>(.*?)</{tag_type}>"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

all_loaded_data_df = pd.DataFrame(all_loaded_data)
all_loaded_data_df['verdict'] = all_loaded_data_df.data.apply(lambda x: extract_verdict_string(x['response']['content'], 'verdict'))
all_loaded_data_df['score'] = all_loaded_data_df.data.apply(lambda x: extract_verdict(x['response']['content'], 'score'))
all_loaded_data_df

In [ ]:
self_count = []
for agent_model in all_loaded_data_df.agent_model.unique():
    for monitor_model in all_loaded_data_df.monitor_model.unique():
        print(agent_model, monitor_model)
        filtered = all_loaded_data_df[(all_loaded_data_df.agent_model == agent_model) & (all_loaded_data_df.monitor_model == monitor_model)]
        if len(filtered) == 0:
            continue
        self_count.append({'agent_model': agent_model, 'monitor_model': monitor_model, 'self_count': len(filtered[filtered.verdict == 'SELF'])/len(filtered)})
self_count_df = pd.DataFrame(self_count)
self_count_df.set_index(['agent_model', 'monitor_model'], inplace=True)
unstacked = self_count_df.unstack()

# Create heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(unstacked['self_count'], annot=True, cmap='YlOrRd', fmt='.2f')
plt.title('Self-Detection Rate by Agent and Monitor Model')
plt.ylabel('Agent Model')
plt.xlabel('Monitor Model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(all_loaded_data_df[all_loaded_data_df.verdict == 'SELF'].score, label = 'Self', alpha = 0.5, density = True)
plt.hist(all_loaded_data_df[all_loaded_data_df.verdict != 'SELF'].score, label = 'Not Self', alpha = 0.5, density = True)
plt.legend()
plt.title('Score Distribution by Verdict')
plt.xlabel('Score')
plt.ylabel('Density')
plt.show()